# HCS Training Curve

Plots the per-restart convergence of the HCS random-restart hill-climb search.

**"Training curve"** here is _not_ a gradient-descent loss curve.  
Each restart runs a full hill-climb from a random starting DAG; the plots track:

| plot | meaning |
|------|---------|
| **BIC-D score per restart** | faint line = that restart's score; bold = running best so far |
| **cn vs restart #** | HCS Good-Turing bound; HCS stops when `cn < hcs_c` (red dashed) |
| **f1/n vs restart #** | fraction of DAG topologies seen exactly once; drops as search revisits modes |
| **Edge count** | complexity (# edges) of each restart's DAG |

Data source: `data/results/paper/mi{50,100}/configs/*/hcs_restarts.jsonl`  
Each JSONL line: `{restart, score, f1, cn, hcs_c, edges}`

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from megavul_diff_analysis.utils.config_utils import find_project_root

In [ ]:
def load_hcs_restarts(jsonl_path: Path) -> pd.DataFrame:
    """Load per-restart records from hcs_restarts.jsonl into a DataFrame."""
    records = []
    with open(jsonl_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            r = json.loads(line)
            records.append({
                'restart': r['restart'],
                'score': r['score'],
                'f1': r['f1'],
                'cn': r['cn'],
                'hcs_c': r['hcs_c'],
                'n_edges': len(r['edges']),
            })
    return pd.DataFrame(records)


project_root = find_project_root()
paper_dir = project_root / 'data' / 'results' / 'paper'

all_configs: list[dict] = []
for mi_dir in sorted(paper_dir.glob('mi*')):
    configs_dir = mi_dir / 'configs'
    if not configs_dir.exists():
        continue
    for config_dir in sorted(configs_dir.glob('*')):
        jsonl = config_dir / 'hcs_restarts.jsonl'
        if not jsonl.exists():
            continue
        result_file = config_dir / 'result.json'
        result = json.loads(result_file.read_text()) if result_file.exists() else {}
        df = load_hcs_restarts(jsonl)
        df['config'] = config_dir.name
        df['mi'] = mi_dir.name
        all_configs.append(dict(name=config_dir.name, mi=mi_dir.name, df=df, result=result))

mi_groups: dict[str, list] = {}
for c in all_configs:
    mi_groups.setdefault(c['mi'], []).append(c)

print(f'Loaded {len(all_configs)} configs across {len(mi_groups)} MI thresholds')

In [ ]:
for mi_label, cfgs in sorted(mi_groups.items()):
    print(f'\n{mi_label}:')
    for c in cfgs:
        r = c['result']
        tag = c['name'].replace(f'{mi_label}_', '')
        n = r.get('n_restarts_used', len(c['df']))
        score = r.get('bic_score') or float('nan')
        do = r.get('n_distinct_optima', '?')
        print(f'  {tag:30s}  restarts={n:3d}  bic={score:12.2f}  distinct_optima={do}')

## BIC-D Score per Restart

Faint line = score from that individual restart's hill-climb.  
Bold line = running best score seen across all restarts up to restart n.

In [ ]:
fig, axes = plt.subplots(1, len(mi_groups), figsize=(7 * len(mi_groups), 5), sharey=False)
if len(mi_groups) == 1:
    axes = [axes]

for ax, (mi_label, cfgs) in zip(axes, sorted(mi_groups.items())):
    colors = plt.cm.tab10(np.linspace(0, 0.9, len(cfgs)))
    for c, color in zip(cfgs, colors):
        df = c['df']
        tag = c['name'].replace(f'{mi_label}_', '')
        ax.plot(df['restart'], df['score'], alpha=0.35, lw=1.0, color=color)
        running_best = df['score'].cummax()
        ax.plot(df['restart'], running_best, lw=1.8, color=color, label=tag)
    ax.set_title(f'MI threshold = {mi_label[2:]}')
    ax.set_xlabel('Restart #')
    ax.set_ylabel('BIC-D score')
    ax.legend(fontsize=7, ncol=2)
    ax.grid(True, alpha=0.25)

fig.suptitle('BIC-D score per restart  (faint = per-restart, bold = running best)', fontsize=12)
plt.tight_layout()
plt.show()

## HCS Convergence: cn vs Restart

$$c_n = \frac{f_1}{n} + C\sqrt{\frac{\ln(3/\delta)}{n}}, \quad C = 2\sqrt{2} + \sqrt{3}$$

where $f_1$ = number of distinct DAG topologies seen exactly once, $n$ = restart count, $\delta$ = confidence parameter.  
HCS stops when $c_n < $ `hcs_c` (red dashed).  A low $c_n$ means the search has likely explored all high-scoring DAG modes.

In [ ]:
fig, axes = plt.subplots(1, len(mi_groups), figsize=(7 * len(mi_groups), 5), sharey=True)
if len(mi_groups) == 1:
    axes = [axes]

for ax, (mi_label, cfgs) in zip(axes, sorted(mi_groups.items())):
    colors = plt.cm.tab10(np.linspace(0, 0.9, len(cfgs)))
    hcs_c_val = None
    for c, color in zip(cfgs, colors):
        df = c['df']
        tag = c['name'].replace(f'{mi_label}_', '')
        ax.plot(df['restart'], df['cn'], lw=1.5, color=color, label=tag)
        if hcs_c_val is None and len(df):
            hcs_c_val = df['hcs_c'].iloc[0]
    if hcs_c_val is not None:
        ax.axhline(hcs_c_val, color='red', ls='--', lw=1.5, label=f'hcs_c = {hcs_c_val}')
    ax.set_title(f'MI threshold = {mi_label[2:]}')
    ax.set_xlabel('Restart #')
    ax.set_ylabel('cn')
    ax.legend(fontsize=7, ncol=2)
    ax.grid(True, alpha=0.25)
    ax.set_ylim(bottom=0)

fig.suptitle('HCS convergence bound cn — stops when cn < hcs_c (red dashed)', fontsize=12)
plt.tight_layout()
plt.show()

## Singleton Fraction f1/n

$f_1/n$ = fraction of all restarts so far that produced a DAG topology seen exactly once.  
As the search saturates its mode space, $f_1/n \to 0$ — the same DAGs keep reappearing.

In [ ]:
fig, axes = plt.subplots(1, len(mi_groups), figsize=(7 * len(mi_groups), 5), sharey=True)
if len(mi_groups) == 1:
    axes = [axes]

for ax, (mi_label, cfgs) in zip(axes, sorted(mi_groups.items())):
    colors = plt.cm.tab10(np.linspace(0, 0.9, len(cfgs)))
    for c, color in zip(cfgs, colors):
        df = c['df']
        tag = c['name'].replace(f'{mi_label}_', '')
        frac = df['f1'] / df['restart']
        ax.plot(df['restart'], frac, lw=1.5, color=color, label=tag)
    ax.set_title(f'MI threshold = {mi_label[2:]}')
    ax.set_xlabel('Restart #')
    ax.set_ylabel('f1 / n')
    ax.legend(fontsize=7, ncol=2)
    ax.grid(True, alpha=0.25)
    ax.set_ylim(0, 1.05)

fig.suptitle('Singleton fraction f1/n per restart', fontsize=12)
plt.tight_layout()
plt.show()

## Edge Count per Restart

How many edges the DAG returned by each restart contains.  Variance here reflects how different random starting DAGs lead to structurally distinct solutions.

In [ ]:
fig, axes = plt.subplots(1, len(mi_groups), figsize=(7 * len(mi_groups), 4), sharey=False)
if len(mi_groups) == 1:
    axes = [axes]

for ax, (mi_label, cfgs) in zip(axes, sorted(mi_groups.items())):
    colors = plt.cm.tab10(np.linspace(0, 0.9, len(cfgs)))
    for c, color in zip(cfgs, colors):
        df = c['df']
        tag = c['name'].replace(f'{mi_label}_', '')
        ax.plot(df['restart'], df['n_edges'], alpha=0.7, lw=1.2, color=color, label=tag)
    ax.set_title(f'MI threshold = {mi_label[2:]}')
    ax.set_xlabel('Restart #')
    ax.set_ylabel('Edge count')
    ax.legend(fontsize=7, ncol=2)
    ax.grid(True, alpha=0.25)

fig.suptitle('Edge count per restart', fontsize=12)
plt.tight_layout()
plt.show()

## Summary Table

From `result.json` for each config.  `converged=True` means HCS stopped before the hard restart cap (100).

In [ ]:
rows = []
for c in all_configs:
    r = c['result']
    final_cn = c['df']['cn'].iloc[-1]
    final_hcs_c = c['df']['hcs_c'].iloc[-1]
    rows.append({
        'config': c['name'],
        'mi': c['mi'],
        'n_restarts': r.get('n_restarts_used'),
        'bic_score': r.get('bic_score'),
        'n_distinct_optima': r.get('n_distinct_optima'),
        'n_edges': r.get('n_edges'),
        'n_edges_on_target': r.get('n_edges_on_target'),
        'elapsed_s': r.get('elapsed_s'),
        'converged': final_cn < final_hcs_c,
    })

summary_df = pd.DataFrame(rows).set_index('config')
summary_df.style.background_gradient(subset=['bic_score', 'n_distinct_optima'], cmap='RdYlGn')